In [1]:
!pip install timm


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip uninstall torch torchvision torchaudio -y


Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing installation: torchvision 0.25.0
Uninstalling torchvision-0.25.0:
  Successfully uninstalled torchvision-0.25.0
Found existing installation: torchaudio 2.10.0
Uninstalling torchaudio-2.10.0:
  Successfully uninstalled torchaudio-2.10.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.


In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

import timm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy import stats

from PIL import Image
from tqdm import tqdm
from collections import OrderedDict

import albumentations as A
from albumentations.pytorch import ToTensorV2


C:\Users\Sarphu\OneDrive - vit.ac.in\Desktop\project1feb17\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
torch.backends.cudnn.benchmark = True

In [5]:
DATA_DIR  = "./teaLeafBDnoAB_healthymodified"
MODEL_DIR = "./tea_models_kfold"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_ARCHITECTURES = [
    "vit_base_patch16_224",
    "swin_base_patch4_window7_224",
    "deit_base_patch16_224",
    "convnext_base"
]

IMG_SIZE   = 224
K_FOLDS    = 5
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 1e-4
SEED       = 42

In [6]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [4]:
def get_train_transform():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),

        # Geometric augmentations
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=30, p=0.5),

        # Photometric augmentations
        A.RandomBrightnessContrast(p=0.5),
        A.GaussianBlur(p=0.2),

        # ImageNet normalization
        A.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        ),

        ToTensorV2()
    ])


def get_val_transform():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),

        A.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        ),

        ToTensorV2()
    ])


In [5]:
class TeaDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        image = np.array(
            Image.open(self.file_paths[idx]).convert("RGB")
        )
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, label

In [6]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction="none")

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        return (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()

In [7]:
def calculate_confidence_interval(data, confidence=0.95):
    n = len(data)
    if n < 2:
        return np.mean(data), 0.0
    mean = np.mean(data)
    se = stats.sem(data)
    h = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, h

In [8]:
class_names = sorted(
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
)

all_file_paths, all_labels = [], []

for idx, cls in enumerate(class_names):
    cls_path = os.path.join(DATA_DIR, cls)
    imgs = [
        os.path.join(cls_path, f)
        for f in os.listdir(cls_path)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]
    all_file_paths.extend(imgs)
    all_labels.extend([idx] * len(imgs))

all_file_paths = np.array(all_file_paths)
all_labels = np.array(all_labels)

num_classes = len(class_names)

In [ ]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
results_summary = []

for arch in MODEL_ARCHITECTURES:
    print(f"\n{'='*40}\nTraining: {arch}\n{'='*40}")

    fold_accs, fold_f1s = [], []

    for fold, (train_idx, val_idx) in enumerate(skf.split(all_file_paths, all_labels)):
        print(f"Fold {fold+1}/{K_FOLDS}")

        model = timm.create_model(
            arch,
            pretrained=True,
            num_classes=num_classes,
            drop_rate=0.3,
            drop_path_rate=0.1
        ).to(device)

        train_ds = TeaDataset(
            all_file_paths[train_idx],
            all_labels[train_idx],
            get_train_transform()
        )
        val_ds = TeaDataset(
            all_file_paths[val_idx],
            all_labels[val_idx],
            get_val_transform()
        )

        labels_train = all_labels[train_idx]
        class_counts = np.bincount(labels_train)
        weights = 1. / torch.tensor(class_counts, dtype=torch.float)
        sample_weights = weights[labels_train]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

        train_loader = DataLoader(
            train_ds, batch_size=BATCH_SIZE, sampler=sampler
        )
        val_loader = DataLoader(
            val_ds, batch_size=BATCH_SIZE, shuffle=False
        )

        criterion = FocalLoss()
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=3
        )

        best_acc = 0.0

        for epoch in tqdm(range(EPOCHS), desc=f"Fold {fold+1} Epochs"):
            model.train()
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(x), y)
                loss.backward()
                optimizer.step()

            model.eval()
            y_true, y_pred = [], []

            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(device), y.to(device)
                    preds = model(x).argmax(1)
                    y_true.extend(y.cpu().numpy())
                    y_pred.extend(preds.cpu().numpy())

            acc = accuracy_score(y_true, y_pred)
            scheduler.step(acc)

            if acc > best_acc:
                best_acc = acc
                torch.save(
                    model.state_dict(),
                    os.path.join(MODEL_DIR, f"{arch}_fold{fold}_best.pth")
                )

        fold_accs.append(best_acc)
        fold_f1s.append(f1_score(y_true, y_pred, average="weighted"))
        print(f"Fold Accuracy: {best_acc:.4f}")

    torch.save(
        model.state_dict(),
        os.path.join(MODEL_DIR, f"{arch}_final.pth")
    )

    acc_mean, acc_ci = calculate_confidence_interval(fold_accs)
    f1_mean, f1_ci   = calculate_confidence_interval(fold_f1s)

    results_summary.append({
        "Model": arch,
        "Accuracy": f"{acc_mean*100:.2f}% ± {acc_ci*100:.2f}%",
        "F1": f"{f1_mean:.4f} ± {f1_ci:.4f}"
    })


Training: vit_base_patch16_224
Fold 1/5


Fold 1 Epochs:  33%|███████████████████████████████████████████▋                                                                                       | 10/30 [28:01<55:29, 166.49s/it]

In [ ]:
results_df = pd.DataFrame(results_summary)
print("\nFinal Cross-Validated Performance")
print(results_df.to_string(index=False))

results_df.to_csv("tea_disease_results_albumentations.csv", index=False)

In [ ]:
# ============================================================
# STACKED ENSEMBLE (ALL MODELS) + LOGISTIC REGRESSION + TTA
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("\n==================== STACKED ENSEMBLE ====================")

# ---------- Load trained base models (Level-0) ----------
def load_trained_model(arch):
    model = timm.create_model(
        arch,
        pretrained=False,
        num_classes=num_classes
    )
    model.load_state_dict(
        torch.load(
            os.path.join(MODEL_DIR, f"{arch}_final.pth"),
            map_location=device
        )
    )
    model.to(device)
    model.eval()
    return model


ensemble_models = {
    arch: load_trained_model(arch)
    for arch in MODEL_ARCHITECTURES
}

# ---------- Meta-dataset (use validation data from last fold) ----------
meta_dataset = TeaDataset(
    all_file_paths[val_idx],
    all_labels[val_idx],
    get_val_transform()
)

meta_loader = DataLoader(
    meta_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ---------- Generate meta-features ----------
X_meta, y_meta = [], []

with torch.no_grad():
    for x, y in meta_loader:
        x = x.to(device)
        probs_all = []

        for model in ensemble_models.values():
            probs = F.softmax(model(x), dim=1)
            probs_all.append(probs.cpu().numpy())

        # Concatenate probabilities: [p1, p2, p3, p4]
        meta_features = np.concatenate(probs_all, axis=1)

        X_meta.append(meta_features)
        y_meta.append(y.numpy())

X_meta = np.vstack(X_meta)          # shape: [N, 4*C]
y_meta = np.concatenate(y_meta)

print("Meta-feature shape:", X_meta.shape)

# ---------- Train Logistic Regression meta-learner ----------
meta_learner = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=1000,
        multi_class="multinomial",
        solver="lbfgs",
        random_state=42
    ))
])

meta_learner.fit(X_meta, y_meta)
print("Meta-learner trained successfully.")

# ---------- Test-Time Augmentation (horizontal flip) ----------
def horizontal_flip(x):
    return torch.flip(x, dims=[3])

# ---------- Final ensemble inference with TTA ----------
test_dataset = TeaDataset(
    all_file_paths,     # replace with true test set if available
    all_labels,
    get_val_transform()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

y_true, y_pred = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        probs_all = []

        for model in ensemble_models.values():
            p_orig = F.softmax(model(x), dim=1)
            p_flip = F.softmax(model(horizontal_flip(x)), dim=1)

            # TTA-averaged probability
            p_tta = 0.5 * (p_orig + p_flip)
            probs_all.append(p_tta.cpu().numpy())

        meta_features = np.concatenate(probs_all, axis=1)
        preds = meta_learner.predict(meta_features)

        y_pred.extend(preds)
        y_true.extend(y.numpy())

# ---------- Final ensemble performance ----------
print("\nSTACKED ENSEMBLE PERFORMANCE (ALL MODELS + TTA)")
print(classification_report(y_true, y_pred, target_names=class_names))
